# 04 - Analisis de sensibilidad | RADAR Cibest

**Fase ASUM-DM:** 5 - Evaluacion  
**Objetivo:** Validar robustez del ranking ante perturbaciones de pesos y comparar TOPSIS vs VIKOR

In [ ]:
import sys
import re
from pathlib import Path
from typing import Optional, Union

import src
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import importlib


from src.utils import load_all_configs, get_variable_catalog, resolve_data_path
from src.scoring.sensitivity import run_sensitivity_analysis, compare_topsis_vikor
from src.scoring.hybrid_scorer import prepare_decision_matrix, run_full_scoring
from src.scoring.weighting import compute_hierarchical_weights
import src.scoring.monte_carlo as monte_carlo

importlib.invalidate_caches()
importlib.reload(monte_carlo)
importlib.reload(src.scoring.hybrid_scorer)
importlib.reload(src.scoring.sensitivity)  



from src.scoring.monte_carlo import (
    coerce_component_series,
    run_monte_carlo_topsis_robustness,
    run_monte_carlo_radar_robustness,
)

configs = load_all_configs()
catalog = get_variable_catalog(configs['variables'])



In [54]:
# ---------------------------------------------------------------------
# Cargar master correcto: master_raw_YYYYMMDD.parquet
# ---------------------------------------------------------------------
raw_dir = resolve_data_path(configs["settings"]["data"]["raw_path"])

pattern = re.compile(r"^master_raw_\d{8}\.parquet$")

master_files = sorted(
    [
        p for p in raw_dir.glob("master_raw_*.parquet")
        if pattern.match(p.name)
    ],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not master_files:
    raise FileNotFoundError(
        "Falta master_raw_YYYYMMDD.parquet. Ejecute primero notebook 01."
    )

master_path = master_files[0]
master = pd.read_parquet(master_path)

print(f"Archivo master cargado: {master_path.name}")
print(f"Master cargado: {master.shape}")
print(f"Variables únicas: {master['variable'].nunique()}")
print("Tiene gdp_growth:", "gdp_growth" in master["variable"].astype(str).str.strip().unique())

# ---------------------------------------------------------------------
# Limpieza defensiva
# ---------------------------------------------------------------------
master = master.copy()
master["variable"] = master["variable"].astype(str).str.strip()

# Eliminar wgi_composite si quedó de versiones anteriores
master = master[master["variable"] != "wgi_composite"].copy()

# ---------------------------------------------------------------------
# Validar matriz de decisión
# ---------------------------------------------------------------------
wide_raw, decision_matrix, excluded = prepare_decision_matrix(master, configs)

print(f"wide_raw: {wide_raw.shape}")
print(f"Decision matrix: {decision_matrix.shape}")
print(f"Países excluidos: {excluded}")

2026-06-12 16:12:20.740 | INFO     | src.data_preparation.cleaning:pivot_long_to_wide:70 - Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)
2026-06-12 16:12:20.746 | WARNING  | src.data_preparation.cleaning:validate_country_coverage:92 - Paises excluidos por >30% variables faltantes: ['CUB']
2026-06-12 16:12:20.855 | INFO     | src.data_preparation.cleaning:impute_missing:145 - Imputacion (regional_median): 107 -> 0 celdas faltantes
2026-06-12 16:12:20.858 | INFO     | src.data_preparation.cleaning:run_cleaning:183 - Limpieza completada: 29 paises x 45 variables
2026-06-12 16:12:20.867 | INFO     | src.data_preparation.feature_engineering:apply_log_transform:73 - Log-transformacion (natural) aplicada a: ['gdp_nominal', 'population_total', 'geographic_distance_km', 'colombian_diaspora_stock', 'stock_market_cap_gdp', 'financial_system_deposits_gdp', 'domestic_credit_private_gdp', 'fdi_net_inflows_gdp', 'personal_remittances_gdp', 'secure_internet_servers_per_million'

Archivo master cargado: master_raw_20260612.parquet
Master cargado: (1281, 5)
Variables únicas: 45
Tiene gdp_growth: True
wide_raw: (29, 46)
Decision matrix: (29, 35)
Países excluidos: ['CUB']


## 1. Sensibilidad de pesos

In [55]:
sens = run_sensitivity_analysis(
    decision_matrix=decision_matrix,
    dimension_weights=configs['weights']['dimension_weights'],
    variable_weights_by_dim=configs['weights']['variable_weights'],
    variable_catalog=catalog,
)
sens

2026-06-12 16:12:21.000 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 29 paises | score max=0.719 (CAN)
2026-06-12 16:12:21.018 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 29 paises | score max=0.725 (CAN)
2026-06-12 16:12:21.035 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 29 paises | score max=0.722 (CAN)
2026-06-12 16:12:21.050 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 29 paises | score max=0.716 (CAN)
2026-06-12 16:12:21.071 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 29 paises | score max=0.715 (USA)
2026-06-12 16:12:21.091 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 29 paises | score max=0.730 (CAN)
2026-06-12 16:12:21.105 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 29 paises | score max=0.725 (CAN)
2026-06-12 16:12:21.122 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 29 paises | score max=0.714 (USA)
2026-06-12 16:12:21.137 | INFO  

,dimension,perturbation,score_corr,topN_overlap,topN_size,countries_in,countries_out
0,macro,0.8,0.9887,10,10,,
1,macro,0.9,0.9931,10,10,,
2,macro,1.1,0.9980,9,10,BRA,JAM
3,macro,1.2,0.9862,9,10,BRA,JAM
4,financial,0.8,0.9966,9,10,BRA,JAM
5,financial,0.9,0.9985,10,10,,
6,financial,1.1,0.9966,10,10,,
7,financial,1.2,0.9911,10,10,,
8,institutional,0.8,0.9901,10,10,,
9,institutional,0.9,0.9966,10,10,,


## 2. Comparacion TOPSIS vs VIKOR

In [56]:
var_weights = compute_hierarchical_weights(
    configs['weights']['dimension_weights'],
    configs['weights']['variable_weights'],
)
var_weights = {k: v for k, v in var_weights.items() if k in decision_matrix.columns}
comparison = compare_topsis_vikor(decision_matrix, var_weights, catalog)
comparison.head(15)

2026-06-12 16:12:21.404 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 29 paises | score max=0.719 (CAN)
2026-06-12 16:12:21.424 | INFO     | src.scoring.ranking:rank:180 - VIKOR completado: 29 paises | v=0.5
2026-06-12 16:12:21.436 | INFO     | src.scoring.sensitivity:compare_topsis_vikor:114 - TOPSIS vs VIKOR: correlacion Spearman = 0.930


,score_topsis,rank_topsis,score_vikor,rank_vikor,rank_diff
country_iso3,,,,,
CAN,0.719156,1,1.000000,1,0
USA,0.717864,2,0.996111,2,0
ESP,0.665248,3,0.960261,3,0
CHL,0.636569,4,0.873094,4,0
URY,0.589495,5,0.758503,5,0
CRI,0.581929,6,0.731742,6,0
BHS,0.557503,7,0.503709,14,7
PAN,0.557322,8,0.727145,7,1
DOM,0.549547,9,0.639350,8,1


## Simulación de Motecarlo

### Preparar IPC y Trending para Monte Carlo

In [57]:
results = run_full_scoring(master, configs, persist=True)

2026-06-12 16:12:21.495 | INFO     | src.data_preparation.cleaning:pivot_long_to_wide:70 - Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)


2026-06-12 16:12:21.503 | WARNING  | src.data_preparation.cleaning:validate_country_coverage:92 - Paises excluidos por >30% variables faltantes: ['CUB']
2026-06-12 16:12:21.622 | INFO     | src.data_preparation.cleaning:impute_missing:145 - Imputacion (regional_median): 107 -> 0 celdas faltantes
2026-06-12 16:12:21.624 | INFO     | src.data_preparation.cleaning:run_cleaning:183 - Limpieza completada: 29 paises x 45 variables
2026-06-12 16:12:21.633 | INFO     | src.data_preparation.feature_engineering:apply_log_transform:73 - Log-transformacion (natural) aplicada a: ['gdp_nominal', 'population_total', 'geographic_distance_km', 'colombian_diaspora_stock', 'stock_market_cap_gdp', 'financial_system_deposits_gdp', 'domestic_credit_private_gdp', 'fdi_net_inflows_gdp', 'personal_remittances_gdp', 'secure_internet_servers_per_million', 'atms_per_100k_adults', 'commercial_bank_branches_per_100k_adults']
2026-06-12 16:12:21.638 | INFO     | src.data_preparation.feature_engineering:run_feature_e

In [58]:
ipc_scores = coerce_component_series(
    results["ipc"],
    value_col=None,
    component_name="ipc",
)

trend_scores = coerce_component_series(
    results["trend"],
    value_col="trend",
    component_name="trend",
)

### Ejecutar Monte Carlo TOPSIS

In [59]:
mc_cfg = configs["settings"].get("monte_carlo", {})

mc_topsis = run_monte_carlo_topsis_robustness(
    decision_matrix=decision_matrix,
    configs=configs,
    variable_catalog=catalog,
    n_simulations=int(mc_cfg.get("n_simulations", 1000)),
    dimension_concentration=float(
        mc_cfg.get("topsis", {}).get("dimension_concentration", 150)
    ),
    variable_concentration=float(
        mc_cfg.get("topsis", {}).get("variable_concentration", 100)
    ),
    random_seed=int(mc_cfg.get("random_seed", 42)),
)

2026-06-12 16:12:22.085 | INFO     | src.scoring.monte_carlo:run_monte_carlo_topsis_robustness:576 - País origen excluido de Monte Carlo TOPSIS: COL
2026-06-12 16:12:22.104 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.787 (CAN)
2026-06-12 16:12:22.127 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.648 (ESP)
2026-06-12 16:12:22.153 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.795 (USA)
2026-06-12 16:12:22.175 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.775 (USA)
2026-06-12 16:12:22.193 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.742 (CAN)
2026-06-12 16:12:22.216 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.725 (CAN)
2026-06-12 16:12:22.239 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.627 (ESP)
2026

### Ejecutar Monte Carlo RADAR

In [61]:
mc_radar = run_monte_carlo_radar_robustness(
    decision_matrix=decision_matrix,
    configs=configs,
    variable_catalog=catalog,
    ipc_scores=ipc_scores,
    trend_scores=trend_scores,
    n_simulations=int(mc_cfg.get("n_simulations", 1000)),
    dimension_concentration=float(
        mc_cfg.get("topsis", {}).get("dimension_concentration", 150)
    ),
    variable_concentration=float(
        mc_cfg.get("topsis", {}).get("variable_concentration", 100)
    ),
    composite_concentration=float(
        mc_cfg.get("radar", {}).get("composite_concentration", 150)
    ),
    perturb_composite_weights=bool(
        mc_cfg.get("radar", {}).get("perturb_composite_weights", True)
    ),
    random_seed=int(mc_cfg.get("random_seed", 42)),
)

2026-06-12 16:14:50.234 | INFO     | src.scoring.monte_carlo:run_monte_carlo_radar_robustness:739 - País origen excluido de Monte Carlo RADAR: COL
2026-06-12 16:14:50.267 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.773 (CAN)
2026-06-12 16:14:50.289 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.665 (ESP)
2026-06-12 16:14:50.314 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.820 (USA)
2026-06-12 16:14:50.336 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.737 (ESP)
2026-06-12 16:14:50.359 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.743 (CAN)
2026-06-12 16:14:50.381 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.727 (CAN)
2026-06-12 16:14:50.402 | INFO     | src.scoring.ranking:rank:133 - TOPSIS completado: 28 paises | score max=0.694 (ESP)
2026-0

In [62]:
# Debería aproximarse a: n_simulations * n_business_lines * n_países_sin_COL
# 1000 * 5 * 28 = 140000
mc_radar["simulation_long"].shape

(140000, 12)

#### Consultas principales

#####  Robustez RADAR por línea

In [63]:
radar_rank_robustness = mc_radar["rank_robustness"]

radar_rank_robustness[
    radar_rank_robustness["business_line"] == "PF"
].sort_values("mean_rank").head(15)

,business_line,country_iso3,mean_rank,median_rank,std_rank,min_rank,max_rank,p10_rank,p90_rank,mean_score,std_score,p10_score,p90_score
112,PF,PAN,1.017,1.0,0.129336,1,2,1.0,1.0,0.700336,0.023151,0.670088,0.728815
113,PF,CRI,2.465,2.0,0.658186,2,4,2.0,3.0,0.640076,0.017124,0.618087,0.660862
114,PF,ESP,3.347,3.0,1.247456,1,10,2.0,5.0,0.628074,0.022373,0.598170,0.655195
115,PF,VEN,4.206,4.0,1.351070,1,13,3.0,6.0,0.614188,0.025920,0.580303,0.647728
116,PF,DOM,4.622,5.0,1.074370,2,8,3.0,6.0,0.605371,0.018993,0.580375,0.629186
117,PF,CHL,6.556,6.0,1.566946,4,14,5.0,8.0,0.578223,0.025266,0.544353,0.609701
118,PF,ECU,7.020,7.0,1.423948,3,15,6.0,9.0,0.570517,0.026958,0.535093,0.604283
119,PF,SLV,8.454,8.0,2.492408,2,20,6.0,12.0,0.553933,0.026617,0.519348,0.586168
120,PF,PER,9.482,9.0,1.168350,8,17,8.0,11.0,0.542108,0.019922,0.515763,0.567506
121,PF,URY,10.195,10.0,2.370780,5,18,8.0,14.0,0.535414,0.020056,0.508585,0.559742


##### Probabilidad Top-N RADAR

In [64]:
radar_topn = mc_radar["topn_probabilities"]

radar_topn[
    radar_topn["business_line"] == "PF"
].sort_values("prob_top_5", ascending=False).head(15)

,business_line,country_iso3,prob_top_3,prob_top_5,prob_top_10,prob_top_15
112,PF,CRI,0.908,1.000,1.000,1.000
113,PF,PAN,1.000,1.000,1.000,1.000
114,PF,ESP,0.647,0.947,1.000,1.000
115,PF,VEN,0.265,0.872,0.999,1.000
116,PF,DOM,0.162,0.817,1.000,1.000
117,PF,CHL,0.000,0.213,0.969,1.000
118,PF,SLV,0.015,0.079,0.803,0.987
119,PF,ECU,0.003,0.069,0.964,1.000
120,PF,URY,0.000,0.002,0.673,0.963
121,PF,GUY,0.000,0.001,0.067,0.327


##### Probabilidad por banda RADAR

In [65]:
radar_tiers = mc_radar["tier_probabilities"]

radar_tiers[
    radar_tiers["business_line"] == "PF"
].sort_values(["Alta", "Media-alta"], ascending=False).head(15)

tier,business_line,country_iso3,Alta,Media-alta,Media,Baja
120,PF,CRI,1.000,0.000,0.000,0.000
131,PF,PAN,1.000,0.000,0.000,0.000
123,PF,ESP,0.947,0.053,0.000,0.000
139,PF,VEN,0.872,0.127,0.001,0.000
121,PF,DOM,0.817,0.183,0.000,0.000
119,PF,CHL,0.213,0.765,0.022,0.000
134,PF,SLV,0.079,0.808,0.106,0.007
122,PF,ECU,0.069,0.916,0.015,0.000
137,PF,URY,0.002,0.801,0.189,0.008
125,PF,GUY,0.001,0.098,0.284,0.617


##### Robustez de correlaciones entre líneas RADAR

In [66]:
mc_radar["line_correlation_robustness"]

,line_a,line_b,mean_spearman,median_spearman,std_spearman,p10_spearman,p90_spearman,min_spearman,max_spearman
0,AD,BD,0.882196,0.885331,0.040759,0.828079,0.932129,0.694581,0.967159
1,PF,BD,0.873338,0.898194,0.086703,0.752053,0.954023,0.357964,0.989053
2,IB,CIB,0.870108,0.873016,0.047722,0.808429,0.929009,0.681445,0.971538
3,AD,CIB,0.792564,0.795293,0.058298,0.711932,0.864258,0.608101,0.932677
4,IB,PF,0.784388,0.793377,0.084669,0.670991,0.886152,0.435687,0.950739
5,IB,AD,0.783620,0.786535,0.059818,0.701697,0.856596,0.592775,0.937603
6,IB,BD,0.778672,0.784893,0.067414,0.686809,0.866995,0.561029,0.917351
7,PF,AD,0.748851,0.762999,0.094679,0.624412,0.853311,0.324576,0.939792
8,BD,CIB,0.746769,0.749863,0.080174,0.637110,0.851122,0.486043,0.926108
9,PF,CIB,0.721955,0.724138,0.100233,0.590586,0.849589,0.399562,0.935961


##### Distribución de alpha/beta/gamma

In [67]:
mc_radar["composite_weight_distribution"]

,alpha,beta,gamma
mean,0.598037,0.301779,0.100184
std,0.040219,0.038243,0.024766
min,0.462461,0.177318,0.035772
max,0.732123,0.425644,0.277558


#### Tabla ejecutiva final

In [73]:
from typing import Optional, Union


def _coerce_base_ranking_to_dataframe(
    base_ranking: Union[pd.DataFrame, pd.Series],
    score_col: Optional[str] = None,
) -> pd.DataFrame:
    """Convierte ranking base RADAR a DataFrame estándar.

    Soporta:
    1. DataFrame con country_iso3, radar_score y rank.
    2. DataFrame indexado por country_iso3.
    3. Series indexada por country_iso3 con score RADAR.

    Returns:
        DataFrame con columnas:
        country_iso3, base_radar_score, base_rank.
    """

    if isinstance(base_ranking, pd.Series):
        base = base_ranking.copy()

        if base.index.name is None:
            base.index.name = "country_iso3"

        base_df = base.rename("base_radar_score").reset_index()

        base_df["base_rank"] = (
            base_df["base_radar_score"]
            .rank(ascending=False, method="min")
            .astype(int)
        )

        return base_df[["country_iso3", "base_radar_score", "base_rank"]]

    base = base_ranking.copy()

    if "country_iso3" not in base.columns:
        base = base.reset_index()

        if "index" in base.columns:
            base = base.rename(columns={"index": "country_iso3"})

    if score_col is None:
        candidate_score_cols = [
            "radar_score",
            "score",
            "final_score",
            "value",
        ]

        score_col = next(
            (col for col in candidate_score_cols if col in base.columns),
            None,
        )

    if score_col is None:
        raise ValueError(
            "No se pudo inferir columna de score en base_ranking. "
            f"Columnas disponibles: {list(base.columns)}"
        )

    if "rank" not in base.columns:
        base["rank"] = (
            base[score_col]
            .rank(ascending=False, method="min")
            .astype(int)
        )

    base = base.rename(
        columns={
            score_col: "base_radar_score",
            "rank": "base_rank",
        }
    )

    return base[["country_iso3", "base_radar_score", "base_rank"]]


def build_mc_executive_table(
    base_ranking: Union[pd.DataFrame, pd.Series],
    mc_rank_robustness: pd.DataFrame,
    mc_topn: pd.DataFrame,
    mc_tiers: pd.DataFrame,
    business_line: str,
    score_col: Optional[str] = None,
) -> pd.DataFrame:
    """Construye tabla ejecutiva de robustez Monte Carlo para una línea.

    Combina:
    - ranking base RADAR;
    - distribución Monte Carlo de rankings;
    - probabilidades Top-N;
    - probabilidades por banda.
    """

    base = _coerce_base_ranking_to_dataframe(
        base_ranking=base_ranking,
        score_col=score_col,
    )

    robustness = mc_rank_robustness[
        mc_rank_robustness["business_line"] == business_line
    ].copy()

    topn = mc_topn[
        mc_topn["business_line"] == business_line
    ].copy()

    tiers = mc_tiers[
        mc_tiers["business_line"] == business_line
    ].copy()

    out = (
        base
        .merge(
            robustness,
            on="country_iso3",
            how="left",
        )
        .merge(
            topn,
            on=["business_line", "country_iso3"],
            how="left",
        )
        .merge(
            tiers,
            on=["business_line", "country_iso3"],
            how="left",
        )
    )

    return out.sort_values("base_rank").reset_index(drop=True)

In [75]:
base_pf = results["radar_by_line"]["PF"]

print(type(base_pf))
print(base_pf.head())
print("index name:", getattr(base_pf.index, "name", None))
print("index dtype:", base_pf.index.dtype)

<class 'pandas.core.series.Series'>
19    0.697524
8     0.638575
9     0.604111
11    0.628794
7     0.578110
Name: PF, dtype: float64
index name: None
index dtype: int64


In [76]:
base_pf_series = results["radar_by_line"]["PF"].copy()

countries_eval = list(decision_matrix_mc.index)

if len(base_pf_series) != len(countries_eval):
    raise ValueError(
        f"La longitud de la serie PF ({len(base_pf_series)}) no coincide "
        f"con el número de países evaluados ({len(countries_eval)})."
    )

base_pf = pd.DataFrame(
    {
        "country_iso3": countries_eval,
        "radar_score": base_pf_series.values,
    }
)

base_pf["country_iso3"] = base_pf["country_iso3"].astype(str).str.strip()

base_pf["rank"] = (
    base_pf["radar_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

base_pf = base_pf.sort_values("rank").reset_index(drop=True)

base_pf.head(15)

,country_iso3,radar_score,rank
0,ARG,0.697524,1
1,BHS,0.638575,2
2,BOL,0.628794,3
3,DOM,0.615954,4
4,BLZ,0.604111,5
5,BRA,0.578110,6
6,BRB,0.568597,7
7,ESP,0.551739,8
8,CAN,0.540742,9
9,GTM,0.534801,10


In [77]:
pf_mc_exec = build_mc_executive_table(
    base_ranking=base_pf,
    mc_rank_robustness=mc_radar["rank_robustness"],
    mc_topn=mc_radar["topn_probabilities"],
    mc_tiers=mc_radar["tier_probabilities"],
    business_line="PF",
    score_col="radar_score",
)

pf_mc_exec.head(15)

,country_iso3,base_radar_score,base_rank,business_line,mean_rank,median_rank,std_rank,min_rank,max_rank,p10_rank,...,p10_score,p90_score,prob_top_3,prob_top_5,prob_top_10,prob_top_15,Alta,Media-alta,Media,Baja
0,ARG,0.697524,1,PF,11.918,11.0,2.898565,6,23,9.0,...,0.492420,0.553788,0.000,0.000,0.335,0.841,0.000,0.611,0.294,0.095
1,BHS,0.638575,2,PF,23.475,24.0,1.947606,10,27,21.0,...,0.416982,0.465745,0.000,0.000,0.001,0.004,0.000,0.001,0.004,0.995
2,BOL,0.628794,3,PF,15.135,15.0,2.726207,9,26,12.0,...,0.473573,0.529021,0.000,0.000,0.022,0.585,0.000,0.053,0.656,0.291
3,DOM,0.615954,4,PF,4.622,5.0,1.074370,2,8,3.0,...,0.580375,0.629186,0.162,0.817,1.000,1.000,0.817,0.183,0.000,0.000
4,BLZ,0.604111,5,PF,26.067,27.0,1.256220,20,27,24.0,...,0.400375,0.441460,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000
5,BRA,0.578110,6,PF,23.668,24.0,2.395311,13,27,20.0,...,0.412132,0.467521,0.000,0.000,0.000,0.005,0.000,0.000,0.009,0.991
6,BRB,0.568597,7,PF,22.778,23.0,2.662669,7,27,20.0,...,0.421993,0.470183,0.000,0.000,0.001,0.014,0.000,0.001,0.025,0.974
7,ESP,0.551739,8,PF,3.347,3.0,1.247456,1,10,2.0,...,0.598170,0.655195,0.647,0.947,1.000,1.000,0.947,0.053,0.000,0.000
8,CAN,0.540742,9,PF,19.489,20.0,3.511155,9,26,14.0,...,0.432715,0.505421,0.000,0.000,0.008,0.156,0.000,0.014,0.205,0.781
9,GTM,0.534801,10,PF,15.620,15.0,3.642921,6,26,11.0,...,0.464891,0.528227,0.000,0.000,0.089,0.510,0.000,0.148,0.458,0.394


In [78]:
print("Filas base:", len(base_pf))
print("Filas tabla ejecutiva:", len(pf_mc_exec))

missing_mc = pf_mc_exec[
    pf_mc_exec["mean_rank"].isna()
]["country_iso3"].tolist()

print("Países sin resultados Monte Carlo:", missing_mc)

Filas base: 28
Filas tabla ejecutiva: 28
Países sin resultados Monte Carlo: []


In [79]:
def build_base_radar_by_line_from_series(
    radar_by_line: dict,
    countries_eval: list,
) -> dict:
    """Convierte results['radar_by_line'] en DataFrames por línea.

    Args:
        radar_by_line: Diccionario {linea: Series} con scores RADAR.
        countries_eval: Lista de países en el mismo orden usado por el scoring.

    Returns:
        Diccionario {linea: DataFrame} con country_iso3, radar_score, rank.
    """

    out = {}

    for business_line, score_series in radar_by_line.items():
        if len(score_series) != len(countries_eval):
            raise ValueError(
                f"La longitud de {business_line} ({len(score_series)}) no coincide "
                f"con países evaluados ({len(countries_eval)})."
            )

        df = pd.DataFrame(
            {
                "country_iso3": countries_eval,
                "radar_score": score_series.values,
            }
        )

        df["country_iso3"] = df["country_iso3"].astype(str).str.strip()

        df["rank"] = (
            df["radar_score"]
            .rank(ascending=False, method="min")
            .astype(int)
        )

        out[business_line] = (
            df
            .sort_values("rank")
            .reset_index(drop=True)
        )

    return out

In [84]:
from typing import Dict, Optional, List

import pandas as pd


def build_base_radar_by_line(
    radar_by_line,
    countries_eval: Optional[List[str]] = None,
    business_lines: Optional[List[str]] = None,
) -> Dict[str, pd.DataFrame]:
    """Convierte results['radar_by_line'] en DataFrames por línea.

    Soporta:
    1. DataFrame ancho:
       country_iso3 | IB | PF | AD | BD | CIB

    2. Dict:
       {linea: Series}

    3. Dict:
       {linea: DataFrame}

    Returns:
        Dict {business_line: DataFrame} con:
        country_iso3, radar_score, rank.
    """

    out: Dict[str, pd.DataFrame] = {}

    # ------------------------------------------------------------------
    # Caso 1: radar_by_line es DataFrame ancho
    # ------------------------------------------------------------------
    if isinstance(radar_by_line, pd.DataFrame):
        df = radar_by_line.copy()

        if "country_iso3" not in df.columns:
            df = df.reset_index()

            if "index" in df.columns:
                df = df.rename(columns={"index": "country_iso3"})

        if "country_iso3" not in df.columns:
            raise ValueError(
                "radar_by_line es DataFrame pero no contiene country_iso3. "
                f"Columnas disponibles: {list(df.columns)}"
            )

        df["country_iso3"] = df["country_iso3"].astype(str).str.strip()

        if business_lines is None:
            business_lines = [
                col for col in df.columns
                if col != "country_iso3"
            ]

        for business_line in business_lines:
            if business_line not in df.columns:
                continue

            tmp = df[["country_iso3", business_line]].copy()
            tmp = tmp.rename(columns={business_line: "radar_score"})

            tmp["radar_score"] = pd.to_numeric(
                tmp["radar_score"],
                errors="coerce",
            )

            tmp = tmp.dropna(subset=["radar_score"])

            tmp["rank"] = (
                tmp["radar_score"]
                .rank(ascending=False, method="min")
                .astype(int)
            )

            out[business_line] = (
                tmp
                .sort_values("rank")
                .reset_index(drop=True)
            )

        return out

    # ------------------------------------------------------------------
    # Caso 2 y 3: radar_by_line es dict
    # ------------------------------------------------------------------
    if isinstance(radar_by_line, dict):
        for business_line, obj in radar_by_line.items():

            # Ignorar llaves que no son líneas de negocio.
            if business_line == "country_iso3":
                continue

            # ----------------------------------------------------------
            # Caso 2A: valor es Series
            # ----------------------------------------------------------
            if isinstance(obj, pd.Series):
                score_series = obj.copy()

                # Si la Series tiene índice no numérico, usarlo como país.
                if not pd.api.types.is_integer_dtype(score_series.index):
                    tmp = (
                        score_series
                        .rename("radar_score")
                        .reset_index()
                    )

                    if tmp.columns[0] != "country_iso3":
                        tmp = tmp.rename(
                            columns={tmp.columns[0]: "country_iso3"}
                        )

                else:
                    # Si la Series tiene índice numérico, necesitamos countries_eval.
                    if countries_eval is None:
                        raise ValueError(
                            f"{business_line} es Series con índice numérico. "
                            "Debe pasar countries_eval o usar un DataFrame con country_iso3."
                        )

                    if len(score_series) != len(countries_eval):
                        raise ValueError(
                            f"La longitud de {business_line} ({len(score_series)}) "
                            f"no coincide con countries_eval ({len(countries_eval)})."
                        )

                    tmp = pd.DataFrame(
                        {
                            "country_iso3": countries_eval,
                            "radar_score": score_series.values,
                        }
                    )

                tmp["country_iso3"] = tmp["country_iso3"].astype(str).str.strip()
                tmp["radar_score"] = pd.to_numeric(
                    tmp["radar_score"],
                    errors="coerce",
                )

                tmp = tmp.dropna(subset=["radar_score"])

                tmp["rank"] = (
                    tmp["radar_score"]
                    .rank(ascending=False, method="min")
                    .astype(int)
                )

                out[business_line] = (
                    tmp
                    .sort_values("rank")
                    .reset_index(drop=True)
                )

            # ----------------------------------------------------------
            # Caso 2B: valor es DataFrame
            # ----------------------------------------------------------
            elif isinstance(obj, pd.DataFrame):
                tmp = obj.copy()

                if "country_iso3" not in tmp.columns:
                    tmp = tmp.reset_index()

                    if "index" in tmp.columns:
                        tmp = tmp.rename(columns={"index": "country_iso3"})

                if "country_iso3" not in tmp.columns:
                    raise ValueError(
                        f"{business_line} es DataFrame pero no contiene country_iso3."
                    )

                candidate_score_cols = [
                    "radar_score",
                    "score",
                    "final_score",
                    business_line,
                    "value",
                ]

                score_col = next(
                    (col for col in candidate_score_cols if col in tmp.columns),
                    None,
                )

                if score_col is None:
                    raise ValueError(
                        f"No se pudo inferir columna score para {business_line}. "
                        f"Columnas: {list(tmp.columns)}"
                    )

                tmp = tmp[["country_iso3", score_col]].copy()
                tmp = tmp.rename(columns={score_col: "radar_score"})

                tmp["country_iso3"] = tmp["country_iso3"].astype(str).str.strip()
                tmp["radar_score"] = pd.to_numeric(
                    tmp["radar_score"],
                    errors="coerce",
                )

                tmp = tmp.dropna(subset=["radar_score"])

                tmp["rank"] = (
                    tmp["radar_score"]
                    .rank(ascending=False, method="min")
                    .astype(int)
                )

                out[business_line] = (
                    tmp
                    .sort_values("rank")
                    .reset_index(drop=True)
                )

            else:
                raise TypeError(
                    f"Tipo no soportado para {business_line}: {type(obj)}"
                )

        return out

    raise TypeError(
        "radar_by_line debe ser DataFrame o dict. "
        f"Tipo recibido: {type(radar_by_line)}"
    )

In [86]:
base_radar_by_line = build_base_radar_by_line(
    radar_by_line=results["radar_by_line"]
)

In [87]:
pf_mc_exec = build_mc_executive_table(
    base_ranking=base_radar_by_line["PF"],
    mc_rank_robustness=mc_radar["rank_robustness"],
    mc_topn=mc_radar["topn_probabilities"],
    mc_tiers=mc_radar["tier_probabilities"],
    business_line="PF",
    score_col="radar_score",
)

pf_mc_exec.head(15)

,country_iso3,base_radar_score,base_rank,business_line,mean_rank,median_rank,std_rank,min_rank,max_rank,p10_rank,...,p10_score,p90_score,prob_top_3,prob_top_5,prob_top_10,prob_top_15,Alta,Media-alta,Media,Baja
0,PAN,0.697524,1,PF,1.017,1.0,0.129336,1,2,1.0,...,0.670088,0.728815,1.000,1.000,1.000,1.000,1.000,0.000,0.000,0.000
1,CRI,0.638575,2,PF,2.465,2.0,0.658186,2,4,2.0,...,0.618087,0.660862,0.908,1.000,1.000,1.000,1.000,0.000,0.000,0.000
2,ESP,0.628794,3,PF,3.347,3.0,1.247456,1,10,2.0,...,0.598170,0.655195,0.647,0.947,1.000,1.000,0.947,0.053,0.000,0.000
3,VEN,0.615954,4,PF,4.206,4.0,1.351070,1,13,3.0,...,0.580303,0.647728,0.265,0.872,0.999,1.000,0.872,0.127,0.001,0.000
4,DOM,0.604111,5,PF,4.622,5.0,1.074370,2,8,3.0,...,0.580375,0.629186,0.162,0.817,1.000,1.000,0.817,0.183,0.000,0.000
5,CHL,0.578110,6,PF,6.556,6.0,1.566946,4,14,5.0,...,0.544353,0.609701,0.000,0.213,0.969,1.000,0.213,0.765,0.022,0.000
6,ECU,0.568597,7,PF,7.020,7.0,1.423948,3,15,6.0,...,0.535093,0.604283,0.003,0.069,0.964,1.000,0.069,0.916,0.015,0.000
7,SLV,0.551739,8,PF,8.454,8.0,2.492408,2,20,6.0,...,0.519348,0.586168,0.015,0.079,0.803,0.987,0.079,0.808,0.106,0.007
8,PER,0.540742,9,PF,9.482,9.0,1.168350,8,17,8.0,...,0.515763,0.567506,0.000,0.000,0.804,0.998,0.000,0.944,0.055,0.001
9,URY,0.534801,10,PF,10.195,10.0,2.370780,5,18,8.0,...,0.508585,0.559742,0.000,0.002,0.673,0.963,0.002,0.801,0.189,0.008


##### Persistencia de resultados

In [88]:
scores_dir = resolve_data_path(configs["settings"]["data"]["scores_path"])
scores_dir.mkdir(parents=True, exist_ok=True)

mc_radar["rank_robustness"].to_parquet(
    scores_dir / "mc_radar_rank_robustness_latest.parquet",
    index=False,
)

mc_radar["topn_probabilities"].to_parquet(
    scores_dir / "mc_radar_topn_probabilities_latest.parquet",
    index=False,
)

mc_radar["tier_probabilities"].to_parquet(
    scores_dir / "mc_radar_tier_probabilities_latest.parquet",
    index=False,
)

mc_radar["line_correlation_robustness"].to_parquet(
    scores_dir / "mc_radar_line_correlation_robustness_latest.parquet",
    index=False,
)

mc_topsis["rank_robustness"].to_parquet(
    scores_dir / "mc_topsis_rank_robustness_latest.parquet",
    index=False,
)